In [1]:
!pip install sacrebleu openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import gc, re, random
import numpy as np
import pandas as pd
from pathlib import Path
import torch
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

random.seed(42)
np.random.seed(42)

DATA_DIR = Path("/kaggle/input/datasets/alexkorablev/qom-corpus")
OUT_DIR  = Path("/kaggle/working/translations")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE = Path("/kaggle/input/notebooks/alexkorablev")

MODELS = {
    "v1_strat":   {
        "es2qom": BASE / "qom-mt-v1-strat/qom-nllb-v1/best-es2qom",
        "qom2es": BASE / "qom-mt-v1-strat/qom-nllb-v1/best-qom2es",
    },
    "v1_nostrat": {
        "es2qom": BASE / "qom-mt-v1-nostrat/qom-nllb-v1-nostrat/best-es2qom",
        "qom2es": BASE / "qom-mt-v1-nostrat/qom-nllb-v1-nostrat/best-qom2es",
    },
    "v2_strat": {
        "es2qom": BASE / "qom-mt-v2-strat-es2qom/qom-nllb-v2-strat-es2qom/best-es2qom",
        "qom2es": BASE / "qom-mt-v2-strat-qom2es/qom-nllb/best-qom2es",
    },
    "v2_nostrat": {
        "es2qom": BASE / "qom-mt-v2-nostrat-es2qom/qom-nllb-v2-nostrat-es2qom/best-es2qom",
        "qom2es": BASE / "qom-mt-v2-nostrat-qom2es/qom-nllb/best-qom2es",
    },
    "bible": {
        "es2qom": BASE / "qom-mt-bible-es2qom/qom-nllb/best-es2qom",
        "qom2es": BASE / "qom-mt-bible-qom2es/qom-nllb/best-qom2es",
    },
}

SRC_LANG = "spa_Latn"
TGT_LANG = "grn_Latn"
MAX_NEW_TOKENS = 128

print("Paths check:")
for name, dirs in MODELS.items():
    for direction, path in dirs.items():
        print(f"  {'OK' if path.exists() else 'MISSING':7} {name}/{direction}")

Paths check:
  OK      v1_strat/es2qom
  OK      v1_strat/qom2es
  OK      v1_nostrat/es2qom
  OK      v1_nostrat/qom2es
  OK      v2_strat/es2qom
  OK      v2_strat/qom2es
  OK      v2_nostrat/es2qom
  OK      v2_nostrat/qom2es
  OK      bible/es2qom
  OK      bible/qom2es


In [3]:
import re as _re
from sklearn.model_selection import GroupShuffleSplit

def read_workbook(path):
    doc_name = path.stem
    xl = pd.ExcelFile(path)
    frames = []
    for sheet in xl.sheet_names:
        try:
            raw = xl.parse(sheet)
        except Exception:
            continue
        raw.columns = [str(c).strip() for c in raw.columns]
        if "linea_qom" not in raw.columns or "linea_es" not in raw.columns:
            continue
        sub = raw[["linea_qom", "linea_es"]].copy()
        sub.columns = ["qom", "es"]
        for col in ["id_linea","id_fragmento","id_seccion","id_capitulo"]:
            sub[col] = raw.get(col, np.nan)
        sub["source_doc"]  = doc_name
        sub["source_path"] = str(path)
        sub["sheet_name"]  = sheet
        frames.append(sub)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

df = pd.concat([read_workbook(p) for p in sorted(DATA_DIR.glob("*.xlsx"))], ignore_index=True)

pp = DATA_DIR / "El Principito.csv"
if pp.exists():
    raw = pd.read_csv(pp)
    raw.columns = [str(c).strip() for c in raw.columns]
    sub = raw[["qom","espanol"]].copy()
    sub.columns = ["qom","es"]
    sub = sub.dropna().query('qom != "" and es != "" and qom != "nan" and es != "nan"')
    sub["id_linea"] = [f"el_principito_{i+1}" for i in range(len(sub))]
    for col in ["id_fragmento","id_seccion","id_capitulo"]:
        sub[col] = np.nan
    sub["source_doc"] = "El Principito"
    sub["source_path"] = str(pp)
    sub["sheet_name"] = "csv"
    df = pd.concat([df, sub], ignore_index=True)

_tok = _re.compile(r"\w+|[^\w\s]", _re.UNICODE)
def ntok(s): return len([t for t in _tok.findall(str(s)) if t.strip()])
df = df.dropna(subset=["qom","es"])
df["qom"] = df["qom"].astype(str).str.strip()
df["es"]  = df["es"].astype(str).str.strip()
df = df[(df["qom"]!="")&(df["es"]!="")&(df["qom"]!="nan")&(df["es"]!="nan")]
df["ql"] = df["qom"].map(ntok); df["el"] = df["es"].map(ntok)
df["lr"] = (df["el"]+1)/(df["ql"]+1)
df = df[(df["ql"]>=2)&(df["el"]>=2)&(df["lr"]>=0.2)&(df["lr"]<=5)]
df = df.drop_duplicates(subset=["qom","es"]).drop(columns=["ql","el","lr"])

def gid(row):
    for col, tag in [("id_fragmento","frag"),("id_seccion","sec"),("id_capitulo","cap")]:
        if pd.notna(row.get(col)):
            return f"{row['source_doc']}__{tag}__{row[col]}"
    return str(row["source_doc"])
df["group_id"] = df.apply(gid, axis=1)

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_train, idx_temp = next(gss1.split(df, groups=df["group_id"]))
temp = df.iloc[idx_temp]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
_, idx_test = next(gss2.split(temp, groups=temp["group_id"]))
test_df = temp.iloc[idx_test].reset_index(drop=True)

print(f"Test set: {len(test_df)} pairs")
print(test_df["source_doc"].value_counts().to_dict())

Test set: 158 pairs
{'Arte verbal qom': 107, 'Materiales del Taller de Lengua y Cultura Toba': 29, 'Educacion Sanitaria Intercultural': 22}


In [4]:
def load_model(path):
    tok   = NllbTokenizer.from_pretrained(str(path), local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(str(path), local_files_only=True)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    return tok, model

def translate_batch(texts, tok, model, src_lang, tgt_lang, batch_size=16):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tok.src_lang = src_lang
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=128)
        if torch.cuda.is_available():
            enc = {k: v.cuda() for k, v in enc.items()}
        with torch.no_grad():
            out = model.generate(
                **enc,
                forced_bos_token_id=tok.convert_tokens_to_ids(tgt_lang),
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=4,
            )
        results.extend(tok.batch_decode(out, skip_special_tokens=True))
    return results

def chrf(hyps, refs):
    return round(sacrebleu.corpus_chrf(hyps, [refs]).score, 2)

In [5]:
qom_sents = test_df["qom"].tolist()
es_sents  = test_df["es"].tolist()

results_qom2es = {"qom": qom_sents, "es_reference": es_sents}
results_es2qom = {"es": es_sents, "qom_reference": qom_sents}
scores = {}

for name, dirs in MODELS.items():
    # QOM → ES
    print(f"[{name}] QOM→ES ...")
    tok, model = load_model(dirs["qom2es"])
    hyp = translate_batch(qom_sents, tok, model, TGT_LANG, SRC_LANG)
    results_qom2es[f"es_{name}"] = hyp
    scores[f"{name}_qom2es"] = chrf(hyp, es_sents)
    print(f"  ChrF++: {scores[f'{name}_qom2es']}")
    del model; gc.collect(); torch.cuda.empty_cache()

    # ES → QOM
    print(f"[{name}] ES→QOM ...")
    tok, model = load_model(dirs["es2qom"])
    hyp = translate_batch(es_sents, tok, model, SRC_LANG, TGT_LANG)
    results_es2qom[f"qom_{name}"] = hyp
    scores[f"{name}_es2qom"] = chrf(hyp, qom_sents)
    print(f"  ChrF++: {scores[f'{name}_es2qom']}")
    del model; gc.collect(); torch.cuda.empty_cache()

[v1_strat] QOM→ES ...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  ChrF++: 30.88
[v1_strat] ES→QOM ...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  ChrF++: 20.31
[bible] ES→QOM ...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  ChrF++: 33.78


In [6]:
df_qom2es = pd.DataFrame(results_qom2es)
df_es2qom = pd.DataFrame(results_es2qom)

p1 = OUT_DIR / "translations_qom2es.csv"
p2 = OUT_DIR / "translations_es2qom.csv"
df_qom2es.to_csv(p1, index=False)
df_es2qom.to_csv(p2, index=False)
print(f"Saved: {p1}")
print(f"Saved: {p2}")

Saved: /kaggle/working/translations/translations_qom2es.csv
Saved: /kaggle/working/translations/translations_es2qom.csv


In [7]:
print("\n" + "="*65)
print("  ALL MODELS — ChrF++ on V1 strat test set")
print("="*65)
print(f"{'model':<15} {'QOM→ES':>10} {'ES→QOM':>10}")
print("-"*65)
for name in MODELS:
    q2e = scores[f"{name}_qom2es"]
    e2q = scores[f"{name}_es2qom"]
    print(f"{name:<15} {q2e:>10.2f} {e2q:>10.2f}")
print("="*65)


  ALL MODELS — ChrF++ on V1 strat test set
model               QOM→ES     ES→QOM
-----------------------------------------------------------------
v1_strat             30.88      38.25
v1_nostrat           33.72      53.62
v2_strat             37.83      56.38
v2_nostrat           46.17      55.01
bible                20.31      33.78
